# Bayesian Parameter Estimation & Model Comparison
**Author:** Dipon Konar  
**Tools:** R (ggplot2, truncnorm, dplyr, reshape2)

## Problem Statement

This project covers two Bayesian inference problems:

1. **Binomial inference** — estimating a probability parameter θ from count data using a truncated uniform prior, deriving the posterior analytically via Bayes' theorem.

2. **Visual word recognition** — comparing a null hypothesis model against a lexical-access model to investigate whether non-word recognition times are systematically longer than word recognition times. Model comparison is done through prior predictive checks and posterior estimation of the effect parameter δ.

---
## Part 1 — Bayesian Inference on a Binomial Parameter

### Model Assumptions

**Likelihood:**  
Data $y$ is modelled as a Binomial draw from 10 trials:
$$L(\theta|y) = \binom{10}{y} \theta^y (1-\theta)^{10-y}$$

**Prior:**  
The parameter $\theta$ is assumed to lie in $[0.5, 1]$ with equal probability — a truncated uniform:
$$p(\theta) = \begin{cases} 2 & 0.5 \leq \theta \leq 1 \\ 0 & \text{otherwise} \end{cases}$$

**Posterior** (via Bayes' rule):
$$p(\theta|y) = \frac{L(\theta|y)\, p(\theta)}{\int L(\theta|y)\, p(\theta)\, d\theta}$$

For $y = 7$, the marginal likelihood works out to $\frac{227}{1408}$.

In [ ]:
library(ggplot2)
library(reshape2)

In [ ]:
# --- likelihood, prior, posterior functions ---

likelihood_binom <- function(theta, y, n = 10) {
  choose(n, y) * theta^y * (1 - theta)^(n - y)
}

prior_truncated <- function(theta) {
  ifelse(theta >= 0.5 & theta <= 1, 2, 0)
}

posterior_binom <- function(theta, y, marginal) {
  likelihood_binom(theta, y) * prior_truncated(theta) / marginal
}

# data + marginal
y_obs       <- 7
marginal_1  <- 227 / 1408

### Posterior at specific θ values

In [ ]:
test_thetas <- c(0.75, 0.25, 1.0)
post_at_points <- sapply(test_thetas, posterior_binom, y = y_obs, marginal = marginal_1)

data.frame(theta = test_thetas, posterior_density = round(post_at_points, 4))

θ = 0.25 and θ = 1 both return 0 — the prior assigns zero mass outside [0.5, 1], so the posterior does too. The density at θ = 0.75 is around 3.1.

### Posterior distribution + MAP estimate

In [ ]:
theta_grid <- seq(0, 1, length.out = 10000)

df_part1 <- data.frame(
  theta          = theta_grid,
  likelihood     = sapply(theta_grid, likelihood_binom, y = y_obs),
  prior          = sapply(theta_grid, prior_truncated),
  posterior      = sapply(theta_grid, posterior_binom, y = y_obs, marginal = marginal_1)
)

map_idx   <- which.max(df_part1$posterior)
map_theta <- df_part1$theta[map_idx]
map_post  <- df_part1$posterior[map_idx]

cat("MAP estimate: theta =", round(map_theta, 3),
    " | posterior density =", round(map_post, 3))

In [ ]:
# posterior plot with MAP
ggplot(df_part1, aes(x = theta, y = posterior)) +
  geom_line(linewidth = 0.8, color = "steelblue") +
  geom_vline(xintercept = map_theta, linetype = "dashed", color = "tomato") +
  geom_point(x = map_theta, y = map_post, color = "tomato", size = 3) +
  annotate("text", x = map_theta + 0.03, y = map_post,
           label = paste0("MAP = (", round(map_theta, 2), ", ", round(map_post, 2), ")"),
           hjust = 0, size = 3.5) +
  labs(title = "Posterior Distribution of θ",
       subtitle = "Binomial likelihood × truncated uniform prior | y = 7, n = 10",
       x = expression(theta), y = "Posterior density") +
  theme_minimal(base_size = 13)

### Likelihood / Prior / Posterior — side by side

In [ ]:
df_long <- melt(df_part1, id.vars = "theta")

ggplot(df_long, aes(x = theta, y = value, color = variable)) +
  geom_line(linewidth = 0.7) +
  facet_wrap(~variable, scales = "free_y", nrow = 3) +
  scale_color_manual(values = c("#E07B54", "#5B8DB8", "#4CAF50")) +
  labs(title = "Likelihood, Prior, and Posterior of θ",
       x = expression(theta), y = "Value") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "none")

The prior truncates all mass below 0.5. The likelihood peaks around 0.7 (the MLE = y/n). After multiplying and normalising, the posterior shifts slightly rightward relative to the likelihood because the prior upweights higher θ values.

---
## Part 2 — Visual Word Recognition: Model Comparison

### Research Question

Does it take longer to recognise non-words than words? Formally, is $\mu_{nw} > \mu_w$?

Two competing models:

| | Null Model | Lexical-Access Model |
|---|---|---|
| $T_w$ | $\mathcal{N}(\mu, \sigma)$ | $\mathcal{N}(\mu, \sigma)$ |
| $T_{nw}$ | $\mathcal{N}(\mu + 0,\ \sigma)$ | $\mathcal{N}(\mu + \delta,\ \sigma)$ |
| $\mu$ prior | $\mathcal{N}(300, 50)$ | $\mathcal{N}(300, 50)$ |
| $\delta$ prior | fixed at 0 | $\mathcal{N}^+(0, 50)$ |
| $\sigma$ | 60 (fixed) | 60 (fixed) |

The truncated normal $\mathcal{N}^+(0, 50)$ constrains δ > 0, encoding the directional hypothesis.

In [ ]:
library(truncnorm)
library(dplyr)

df <- read.table("recognition.csv", sep = ",", header = TRUE)[, -1]
Tw  <- df$Tw
Tnw <- df$Tnw
n   <- length(Tw)
SIGMA <- 60

cat("N =", n, "\n")
cat("Mean Tw =",  round(mean(Tw), 2),  "| SD =", round(sd(Tw), 2), "\n")
cat("Mean Tnw =", round(mean(Tnw), 2), "| SD =", round(sd(Tnw), 2), "\n")
cat("Observed difference (Tnw - Tw) =", round(mean(Tnw) - mean(Tw), 2), "ms\n")

### Prior predictive simulation

In [ ]:
set.seed(42)
N_SIM <- 2000

simulate_model <- function(n_sim, n_obs, sigma, model = c("null", "lex")) {
  model  <- match.arg(model)
  mu_s   <- rnorm(n_sim, 300, 50)
  delta_s <- if (model == "lex") rtruncnorm(n_sim, a = 0, b = Inf, mean = 0, sd = 50)
             else rep(0, n_sim)

  out <- data.frame(
    sample = rep(1:n_sim, each = n_obs),
    mu     = rep(mu_s, each = n_obs),
    delta  = rep(delta_s, each = n_obs)
  )
  out$Tw_sim  <- rnorm(nrow(out), mean = out$mu, sd = sigma)
  out$Tnw_sim <- rnorm(nrow(out), mean = out$mu + out$delta, sd = sigma)
  out
}

sim_null <- simulate_model(N_SIM, n, SIGMA, "null")
sim_lex  <- simulate_model(N_SIM, n, SIGMA, "lex")

In [ ]:
# summarise mean differences per simulated dataset
summarise_diff <- function(sim_df) {
  sim_df %>%
    group_by(sample) %>%
    summarise(diff = mean(Tnw_sim) - mean(Tw_sim), .groups = "drop")
}

null_summary <- summarise_diff(sim_null)
lex_summary  <- summarise_diff(sim_lex)
diff_obs     <- mean(Tnw) - mean(Tw)

In [ ]:
# prior predictive check — predicted mean differences
ggplot() +
  geom_histogram(data = null_summary, aes(x = diff, fill = "Null"),
                 alpha = 0.6, bins = 40) +
  geom_histogram(data = lex_summary,  aes(x = diff, fill = "Lexical-access"),
                 alpha = 0.6, bins = 40) +
  geom_vline(xintercept = diff_obs, color = "red", linetype = "dashed", linewidth = 1) +
  annotate("text", x = diff_obs + 2, y = Inf, vjust = 2,
           label = paste0("Observed = ", round(diff_obs, 1), " ms"),
           color = "red", size = 3.5) +
  scale_fill_manual(values = c("Null" = "#5B8DB8", "Lexical-access" = "#F4A460")) +
  labs(title = "Prior Predictive Check",
       subtitle = "Distribution of predicted mean difference (Tnw − Tw)",
       x = "Predicted mean difference (ms)",
       y = "Count", fill = "Model") +
  theme_minimal(base_size = 13)

The null model is centred at 0 — it can't predict a systematic positive difference. The lexical-access model is shifted right, and the observed difference (red dashed line) falls well within its predictive range, but at the edge of the null model's range. This already hints the lexical-access model is more consistent with the data.

### Posterior of μ — Null hypothesis model

In [ ]:
mu_grid <- seq(200, 400, length.out = 1000)

# pool both conditions under null (same mu for Tw and Tnw)
log_post_mu <- sapply(mu_grid, function(m) {
  sum(dnorm(Tw,  m, SIGMA, log = TRUE)) +
  sum(dnorm(Tnw, m, SIGMA, log = TRUE)) +
  dnorm(m, 300, 50, log = TRUE)
})

post_mu <- exp(log_post_mu - max(log_post_mu))

df_mu <- data.frame(mu = mu_grid, posterior = post_mu)

ggplot(df_mu, aes(x = mu, y = posterior)) +
  geom_line(color = "steelblue", linewidth = 0.9) +
  labs(title = "Unnormalised Posterior of μ — Null Model",
       x = expression(mu), y = "Unnormalised posterior") +
  theme_minimal(base_size = 13)

### Posterior of δ — Lexical-access model

δ is marginalised over μ numerically. For each δ value we integrate out μ using a grid.

In [ ]:
delta_grid <- seq(0, 200, length.out = 200)
mu_grid2   <- seq(200, 400, length.out = 200)

log_marg_delta <- sapply(delta_grid, function(d) {
  log_like_mu <- sapply(mu_grid2, function(m) {
    sum(dnorm(Tw,  m,   SIGMA, log = TRUE)) +
    sum(dnorm(Tnw, m+d, SIGMA, log = TRUE))
  })
  log_prior_mu <- dnorm(mu_grid2, 300, 50, log = TRUE)
  log_prior_d  <- log(dtruncnorm(d, a = 0, b = Inf, mean = 0, sd = 50))
  lp <- log_like_mu + log_prior_mu + log_prior_d
  log(sum(exp(lp - max(lp)))) + max(lp)  # stable log-sum-exp
})

post_delta <- exp(log_marg_delta - max(log_marg_delta))

df_delta <- data.frame(delta = delta_grid, posterior = post_delta)

In [ ]:
ggplot(df_delta, aes(x = delta, y = posterior)) +
  geom_line(color = "#E07B54", linewidth = 0.9) +
  geom_vline(xintercept = diff_obs, linetype = "dashed", color = "steelblue") +
  annotate("text", x = diff_obs + 3, y = 0.6,
           label = paste0("Observed\ndiff = ", round(diff_obs, 1), " ms"),
           color = "steelblue", size = 3.5, hjust = 0) +
  labs(title = "Marginalised Posterior of δ — Lexical-access Model",
       subtitle = "δ = mean non-word RT − mean word RT",
       x = expression(delta ~ "(ms)"), y = "Unnormalised posterior") +
  theme_minimal(base_size = 13)

### Closed-form posterior of μ (Normal-Normal conjugacy check)

In [ ]:
# under the null, pooling Tw and Tnw, the conjugate posterior is available analytically
y_all   <- c(Tw, Tnw)
n_total <- length(y_all)
mu0     <- 300; sigma0 <- 50

sigma_post <- 1 / sqrt((1/sigma0^2) + (n_total/SIGMA^2))
mu_post    <- sigma_post^2 * ((mu0/sigma0^2) + (sum(y_all)/SIGMA^2))

cat(sprintf("Posterior mu:    %.3f\n", mu_post))
cat(sprintf("Posterior sigma: %.3f\n", sigma_post))

post_samples_analytic <- rnorm(10000, mu_post, sigma_post)

ggplot(data.frame(mu = post_samples_analytic), aes(x = mu)) +
  geom_histogram(bins = 50, fill = "steelblue", color = "white", alpha = 0.8) +
  labs(title = "Analytic Posterior Samples — μ (Null Model)",
       x = expression(mu), y = "Count") +
  theme_minimal(base_size = 13)

---
## Results & Conclusions

**Part 1**
- The MAP estimate for θ is ~0.70, close to the MLE (y/n = 0.7), slightly pulled right by the prior.
- Values outside [0.5, 1] have zero posterior mass — the prior is informative and excludes them entirely.

**Part 2**
- The observed mean difference is positive (non-words are slower), which is outside the bulk of the null model's prior predictive distribution but sits comfortably within the lexical-access model's predictions.
- The posterior of δ peaks away from zero, consistent with a genuine word-frequency / lexical access effect.
- The null model (δ = 0) struggles to account for the data; the lexical-access model provides a better description.

> **Interpretation:** There is evidence that non-word recognition takes longer than word recognition, consistent with the lexical-access hypothesis — words benefit from stored lexical representations, speeding up recognition relative to novel non-word strings.